# Chapter 1.2: User-Item Interaction Data

## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish between explicit and implicit feedback and their modeling implications
2. Construct and manipulate user-item interaction matrices
3. Analyze data sparsity and its effects on recommendation quality
4. Understand the cold-start problem from both user and item perspectives
5. Implement and compare different data splitting strategies (random, temporal, leave-one-out)
6. Visualize interaction patterns and long-tail distributions
7. Recognize and handle common data quality issues in recommendation data

## Prerequisites

- Chapter 1.1: The Recommendation Problem
- Basic NumPy and matplotlib skills

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part1/chapter_1.2_interaction_data.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part1/chapter_1.2_interaction_data.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from collections import defaultdict, Counter

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

print("All imports successful!")

## 1. Explicit vs Implicit Feedback

### Explicit Feedback
Users directly express preferences:
- Star ratings (1-5 on Netflix, Amazon)
- Thumbs up/down (YouTube)
- Written reviews

The interaction matrix $R \in \mathbb{R}^{|U| \times |I|}$ where $R_{ui}$ is the rating user $u$ gave to item $i$.

### Implicit Feedback
Preferences inferred from behavior:
- Clicks, views, purchases, dwell time
- Add to cart, add to wishlist
- Skips, scroll-past

The interaction matrix $Y \in \{0, 1\}^{|U| \times |I|}$ where $Y_{ui} = 1$ indicates an observed interaction.

| Property | Explicit | Implicit |
|---|---|---|
| **Signal clarity** | Clear preference | Ambiguous |
| **Volume** | Sparse | Dense (relatively) |
| **Negative signal** | Low ratings | Absence = unknown, NOT dislike |
| **User effort** | High (must rate) | Zero (passive) |

> **⚠️ Common Pitfall:** With implicit data, a missing entry does NOT mean the user dislikes the item. The user may simply not know it exists. This fundamentally changes how we define negative samples.

In [ ]:
def generate_synthetic_movielens(n_users=500, n_items=200, density=0.05, seed=42):
    """
    Generate synthetic MovieLens-style interaction data.
    
    Simulates realistic patterns:
    - Power-law item popularity
    - Power-law user activity 
    - Rating bias per user
    """
    np.random.seed(seed)
    
    # Item popularity follows power law (few blockbusters, many niche items)
    item_popularity = np.random.power(0.3, n_items)
    item_popularity /= item_popularity.sum()
    
    # User activity follows power law
    user_activity = np.random.power(0.5, n_users)
    n_ratings_per_user = (user_activity * n_items * density * 5).astype(int).clip(1, n_items // 2)
    
    # User bias: some users rate high, some low
    user_bias = np.random.normal(3.5, 0.5, n_users)
    
    users, items, ratings, timestamps = [], [], [], []
    base_time = 1000000
    
    for u in range(n_users):
        n_u = n_ratings_per_user[u]
        # Sample items based on popularity
        rated_items = np.random.choice(n_items, size=n_u, replace=False, p=item_popularity)
        for i in rated_items:
            users.append(u)
            items.append(i)
            r = np.clip(np.random.normal(user_bias[u], 1.0), 1, 5)
            ratings.append(round(r * 2) / 2)  # Round to nearest 0.5
            timestamps.append(base_time + np.random.randint(0, 100000))
    
    return {
        'user_id': np.array(users),
        'item_id': np.array(items),
        'rating': np.array(ratings),
        'timestamp': np.array(timestamps)
    }


# Generate data
data = generate_synthetic_movielens()
n_users = len(np.unique(data['user_id']))
n_items = len(np.unique(data['item_id']))
n_interactions = len(data['rating'])

print(f"Generated dataset:")
print(f"  Users: {n_users}")
print(f"  Items: {n_items}")
print(f"  Interactions: {n_interactions}")
print(f"  Density: {n_interactions / (n_users * n_items):.4%}")
print(f"  Rating range: [{data['rating'].min()}, {data['rating'].max()}]")
print(f"  Mean rating: {data['rating'].mean():.2f}")

## 2. The Interaction Matrix

The user-item interaction matrix is the fundamental data structure in recommendation systems:

$$
R = \begin{pmatrix}
r_{11} & ? & r_{13} & ? & \cdots \\
? & r_{22} & ? & r_{24} & \cdots \\
r_{31} & ? & ? & ? & \cdots \\
\vdots & & & & \ddots
\end{pmatrix}
$$

where $?$ denotes missing entries. The key challenge: **most entries are missing**.

In [ ]:
# Build the interaction matrix (sparse)
interaction_matrix = sparse.csr_matrix(
    (data['rating'], (data['user_id'], data['item_id'])),
    shape=(500, 200)
)

# Visualize a small portion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dense view of first 30 users x 30 items
small_matrix = interaction_matrix[:30, :30].toarray()
im = axes[0].imshow(small_matrix, cmap='YlOrRd', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Item ID')
axes[0].set_ylabel('User ID')
axes[0].set_title('Interaction Matrix (30x30 subset)\n(0 = no interaction)')
plt.colorbar(im, ax=axes[0], label='Rating')

# Sparsity pattern of full matrix
# Sample for visualization since full matrix is too large
axes[1].spy(interaction_matrix[:100, :100], markersize=1, color='steelblue')
axes[1].set_xlabel('Item ID')
axes[1].set_ylabel('User ID')
axes[1].set_title('Sparsity Pattern (100x100 subset)\n(dots = observed interactions)')

plt.tight_layout()
plt.show()

## 3. Data Sparsity Analysis

Real-world recommendation datasets are extremely sparse:

| Dataset | Users | Items | Interactions | Density |
|---|---|---|---|---|
| MovieLens 1M | 6K | 4K | 1M | 4.2% |
| Amazon Reviews | 21M | 10M | 83M | 0.000004% |
| Netflix Prize | 480K | 18K | 100M | 1.2% |

Sparsity creates challenges:
- Insufficient data for reliable similarity computation
- Matrix factorization prone to overfitting
- Long-tail items have very few interactions

> **💡 Concept:** The "long tail" phenomenon means that a small fraction of items account for most interactions, while the vast majority of items have very few. A good recommendation system should serve the long tail too.

In [ ]:
# Analyze sparsity patterns
items_per_user = np.diff(interaction_matrix.indptr)  # Number of rated items per user
users_per_item = np.diff(interaction_matrix.T.tocsr().indptr)  # Number of users per item

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# User activity distribution (log scale)
axes[0].hist(items_per_user, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Number of Rated Items')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('User Activity Distribution')
axes[0].axvline(np.median(items_per_user), color='red', linestyle='--', 
                label=f'Median={np.median(items_per_user):.0f}')
axes[0].legend()

# Item popularity distribution (sorted)
sorted_popularity = np.sort(users_per_item)[::-1]
axes[1].plot(range(len(sorted_popularity)), sorted_popularity, color='coral', linewidth=2)
axes[1].fill_between(range(len(sorted_popularity)), sorted_popularity, alpha=0.3, color='coral')
axes[1].set_xlabel('Item Rank (by popularity)')
axes[1].set_ylabel('Number of Interactions')
axes[1].set_title('Item Popularity: Long-Tail Distribution')

# Mark the 80/20 point
cumsum = np.cumsum(sorted_popularity)
threshold_idx = np.searchsorted(cumsum, 0.8 * cumsum[-1])
axes[1].axvline(threshold_idx, color='red', linestyle='--',
                label=f'Top {threshold_idx} items = 80% interactions')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"User stats: mean={items_per_user.mean():.1f}, median={np.median(items_per_user):.1f}, "
      f"max={items_per_user.max()}, min={items_per_user.min()}")
print(f"Item stats: mean={users_per_item.mean():.1f}, median={np.median(users_per_item):.1f}, "
      f"max={users_per_item.max()}, min={users_per_item.min()}")
print(f"\nLong-tail: Top {threshold_idx}/{len(sorted_popularity)} items "
      f"({threshold_idx/len(sorted_popularity):.1%}) account for 80% of interactions")

## 4. The Cold-Start Problem

Cold start occurs when the system lacks sufficient interaction data:

### User Cold Start
A new user joins with no interaction history.
- CF cannot find similar users
- Solutions: ask for initial preferences, use demographics, popularity-based fallback

### Item Cold Start  
A new item is added with no interactions.
- CF cannot compute item similarities
- Solutions: use item content features, editorial boosting, exploration strategies

### System Cold Start
An entirely new system with no data.
- Must bootstrap from scratch
- Solutions: import data, content-based, knowledge-based

> **⚠️ Common Pitfall:** Even after solving cold start, there's a "warm start" phase where limited data leads to noisy recommendations. Don't evaluate a model on users with very few interactions and expect good results.

In [ ]:
# Simulate cold start: how does recommendation quality change with interaction count?
np.random.seed(42)

def cosine_similarity_matrix(matrix):
    """Compute pairwise cosine similarity."""
    norms = np.sqrt((matrix ** 2).sum(axis=1)).A1
    norms[norms == 0] = 1  # Avoid division by zero
    normalized = matrix.multiply(1 / norms[:, np.newaxis])
    return normalized.dot(normalized.T).toarray()

# Group users by activity level
activity_bins = [(1, 3), (4, 7), (8, 15), (16, 30), (31, 100)]
bin_labels = ['1-3', '4-7', '8-15', '16-30', '31+']

# For each group, compute average number of non-zero similarities
binary_matrix = (interaction_matrix > 0).astype(float)
user_sim = cosine_similarity_matrix(binary_matrix)
np.fill_diagonal(user_sim, 0)

avg_sim_by_group = []
meaningful_neighbors = []  # Neighbors with sim > 0.1

for low, high in activity_bins:
    mask = (items_per_user >= low) & (items_per_user <= high)
    if mask.sum() > 0:
        avg_sim_by_group.append(user_sim[mask].mean())
        meaningful_neighbors.append((user_sim[mask] > 0.1).sum(axis=1).mean())
    else:
        avg_sim_by_group.append(0)
        meaningful_neighbors.append(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(bin_labels, avg_sim_by_group, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('User Activity (# ratings)')
axes[0].set_ylabel('Average Cosine Similarity')
axes[0].set_title('User Similarity vs Activity Level\n(Cold-Start Effect)')

axes[1].bar(bin_labels, meaningful_neighbors, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('User Activity (# ratings)')
axes[1].set_ylabel('Avg # Meaningful Neighbors (sim>0.1)')
axes[1].set_title('Useful Neighbors vs Activity Level')

plt.tight_layout()
plt.show()

## 5. Data Splitting Strategies

How you split data for train/test dramatically affects evaluation validity.

### Random Split
Randomly assign interactions to train/test. Simple but ignores temporal ordering.

### Temporal Split
Split by time: train on past, test on future. Most realistic for production.

$$
\text{Train}: \{(u, i, r, t) \mid t < t_{\text{split}}\}, \quad
\text{Test}: \{(u, i, r, t) \mid t \geq t_{\text{split}}\}
$$

### Leave-One-Out (LOO)
For each user, hold out the **last interaction** (by time) for testing.

> **🔑 Pro Tip:** Always use temporal split for final evaluation in production settings. Random split causes "data leakage from the future" - you train on interactions that happen after the test interactions, which inflates metrics.

In [ ]:
def random_split(data, test_ratio=0.2, seed=42):
    """Random train/test split."""
    np.random.seed(seed)
    n = len(data['rating'])
    indices = np.random.permutation(n)
    split_point = int(n * (1 - test_ratio))
    
    train_idx = indices[:split_point]
    test_idx = indices[split_point:]
    
    train = {k: v[train_idx] for k, v in data.items()}
    test = {k: v[test_idx] for k, v in data.items()}
    return train, test


def temporal_split(data, test_ratio=0.2):
    """Temporal train/test split."""
    sorted_idx = np.argsort(data['timestamp'])
    split_point = int(len(sorted_idx) * (1 - test_ratio))
    
    train_idx = sorted_idx[:split_point]
    test_idx = sorted_idx[split_point:]
    
    train = {k: v[train_idx] for k, v in data.items()}
    test = {k: v[test_idx] for k, v in data.items()}
    return train, test


def leave_one_out_split(data):
    """Leave-one-out split: hold out each user's last interaction."""
    n = len(data['rating'])
    
    # Find the last interaction for each user (by timestamp)
    user_last_idx = {}
    for idx in range(n):
        u = data['user_id'][idx]
        t = data['timestamp'][idx]
        if u not in user_last_idx or t > data['timestamp'][user_last_idx[u]]:
            user_last_idx[u] = idx
    
    test_indices = set(user_last_idx.values())
    train_idx = np.array([i for i in range(n) if i not in test_indices])
    test_idx = np.array(list(test_indices))
    
    train = {k: v[train_idx] for k, v in data.items()}
    test = {k: v[test_idx] for k, v in data.items()}
    return train, test


# Compare splits
random_train, random_test = random_split(data)
temporal_train, temporal_test = temporal_split(data)
loo_train, loo_test = leave_one_out_split(data)

print("Split Comparison:")
print(f"{'Method':<15} {'Train Size':>12} {'Test Size':>12} {'Test Users':>12} {'Test Items':>12}")
print("-" * 65)
for name, tr, te in [('Random', random_train, random_test),
                      ('Temporal', temporal_train, temporal_test),
                      ('Leave-One-Out', loo_train, loo_test)]:
    print(f"{name:<15} {len(tr['rating']):>12} {len(te['rating']):>12} "
          f"{len(np.unique(te['user_id'])):>12} {len(np.unique(te['item_id'])):>12}")

In [ ]:
# Visualize the data leakage issue with random split
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random split: test timestamps are mixed with train
axes[0].scatter(range(len(random_train['timestamp'])), 
                np.sort(random_train['timestamp']), 
                s=1, alpha=0.3, label='Train', color='steelblue')
axes[0].scatter(range(len(random_test['timestamp'])),
                np.sort(random_test['timestamp']),
                s=1, alpha=0.3, label='Test', color='coral')
axes[0].set_xlabel('Interaction Index (sorted by time)')
axes[0].set_ylabel('Timestamp')
axes[0].set_title('Random Split\n(Test overlaps with Train in time!)')
axes[0].legend(markerscale=10)

# Temporal split: clean separation
axes[1].scatter(range(len(temporal_train['timestamp'])),
                np.sort(temporal_train['timestamp']),
                s=1, alpha=0.3, label='Train', color='steelblue')
axes[1].scatter(range(len(temporal_test['timestamp'])),
                np.sort(temporal_test['timestamp']),
                s=1, alpha=0.3, label='Test', color='coral')
axes[1].set_xlabel('Interaction Index (sorted by time)')
axes[1].set_ylabel('Timestamp')
axes[1].set_title('Temporal Split\n(Clean past/future separation)')
axes[1].legend(markerscale=10)

plt.tight_layout()
plt.show()

> **💡 Concept:** Notice how the random split mixes timestamps between train and test. This means the model can "cheat" by learning from future interactions during training. Temporal split prevents this leakage.

---

## Exercises

### 🏋️ Exercise 1: Build and Analyze a Synthetic Interaction Matrix

Generate a more realistic interaction matrix with user segments (e.g., active vs casual users) and item categories.

In [ ]:
def generate_segmented_data(n_users=300, n_items=150, n_categories=5, seed=42):
    """
    Generate data with user segments and item categories.
    Each user segment has preference for certain categories.
    
    Returns:
        interaction_matrix: sparse matrix
        user_segments: array of segment labels
        item_categories: array of category labels
    """
    # TODO: Implement this function
    # 1. Assign each user to a segment (0, 1, or 2)
    # 2. Assign each item to a category (0 to n_categories-1)
    # 3. Create a segment-category preference matrix (e.g., segment 0 prefers categories 0,1)
    # 4. Generate interactions based on these preferences
    # 5. Return the sparse interaction matrix, user segments, and item categories
    pass


# TODO: Call generate_segmented_data and visualize:
# 1. The interaction matrix (reorder rows/columns by segment/category to see block structure)
# 2. Sparsity per segment and per category
# 3. Distribution of ratings per segment

### 🏋️ Exercise 2: Implement Temporal Split with Validation Set

Implement a three-way temporal split: train / validation / test.

In [ ]:
def temporal_split_3way(data, val_ratio=0.1, test_ratio=0.1):
    """
    Three-way temporal split.
    
    Args:
        data: dict with 'user_id', 'item_id', 'rating', 'timestamp'
        val_ratio: fraction for validation
        test_ratio: fraction for test
    
    Returns:
        train, val, test dicts
    """
    # TODO: Implement this function
    # Sort by timestamp, then split into three consecutive parts
    pass


# TODO: Test your split and verify:
# 1. No temporal overlap between splits
# 2. All test users appear in training data (for fair evaluation)
# 3. Print statistics about each split

### 🏋️ Exercise 3: Cold-Start Analysis

Analyze how many users/items in the test set are "cold" (have zero or few training interactions).

In [ ]:
# TODO: Using the temporal split from above:
# 1. Count users in test that have 0, 1-3, 4-10, 10+ interactions in training
# 2. Count items in test that have 0, 1-3, 4-10, 10+ interactions in training
# 3. Create a visualization showing the cold-start distribution
# 4. Discuss: How would you handle the cold-start users/items?

# Hint: use Counter on user_id/item_id from training data
pass

### 🏋️ Exercise 4: Implicit Feedback Conversion

Convert explicit ratings to implicit feedback and analyze the differences.

In [ ]:
# TODO: 
# 1. Convert ratings to binary: rating >= 3.5 -> positive, else negative
# 2. Try another threshold (e.g., >= 4.0) and compare
# 3. Create an "implicit" version where all interactions are positive (ignoring ratings)
# 4. For each version, compute and compare:
#    - Density of positive interactions
#    - Positive/negative ratio per user
#    - How many items switch from positive to negative between thresholds
# 5. Visualize the differences
pass

## Summary

In this notebook, we covered:

1. **Explicit vs implicit feedback**: Ratings vs behavioral signals, with fundamentally different modeling implications
2. **Interaction matrices**: Sparse representations and their visualization
3. **Data sparsity**: Long-tail distributions and their impact on recommendation quality
4. **Cold-start problem**: Three variants (user, item, system) and mitigation strategies
5. **Data splitting**: Random, temporal, and leave-one-out strategies with temporal being preferred for production

### Key Takeaways

- Most real-world data is implicit, not explicit
- Missing entries in implicit data are NOT negatives - they're unknowns
- Always use temporal splitting for realistic evaluation
- The long-tail is where most recommendation value lies

### Next Up

In **Chapter 1.3**, we'll implement classical collaborative filtering algorithms: UserCF, ItemCF, and Matrix Factorization.